# Phase 4: Station reliability

In this notebook we count how many train disruptions each station had in 2021–2023. We use this as a measure of how reliable a station is. The result is one row per station, which we later link to the neighbourhoods (Phase 5).


In [1]:
from pathlib import Path
import pandas as pd

PROCESSED = Path('data/cleaned_data')

# Load the cleaned data from Phase 1
storingen = pd.read_pickle(PROCESSED / 'storingen.pkl')
stations = pd.read_pickle(PROCESSED / 'stations.pkl')

print(len(storingen))


15541


In [3]:
# Remove disruptions without a station code
storingen = storingen[storingen['rdt_station_codes'].notna()]

# Remove planned engineering work
storingen = storingen[storingen['cause_group'] != 'engineering work']

print(len(storingen))

storingen.head()

14928


,rdt_id,ns_lines,rdt_lines,rdt_lines_id,rdt_station_names,rdt_station_codes,cause_nl,cause_en,statistical_cause_nl,statistical_cause_en,cause_group,start_time,end_time,duration_minutes
0,35626,Hengelo-Bielefeld,"Berlin Ostbahnhof - Hengelo, Bielefeld Hbf - H...","91,124,174","Hengelo,Hengelo Oost,Oldenzaal","HGL, HGLO, ODZ",defecte trein,broken down train,defecte trein,broken down train,rolling stock,2021-01-01 08:12:55,2021-01-01 11:30:00,197.0
1,35627,Amsterdam-Almere; Amsterdam-Amersfoort,"Amersfoort - Amsterdam Centraal, Amsterdam Cen...","135,145","Amsterdam Centraal,Amsterdam Muiderpoort,Amste...","ASD, ASDM, ASSP, DMN, WP",defecte trein,broken down train,defecte trein,broken down train,rolling stock,2021-01-01 10:17:19,2021-01-01 10:29:38,12.0
2,35628,Utrecht-Baarn,Baarn - Utrecht Centraal,44,"Baarn,Den Dolder,Soestdijk,Soest,Soest Zuid","BRN, DLD, SD, ST, STZ",defecte bovenleiding,damaged overhead wires,defecte bovenleiding,damaged overhead wires,infrastructure,2021-01-01 10:44:22,2021-01-01 17:44:52,421.0
3,35629,Zwolle-Emmen,Emmen - Zwolle,159,"Gramsbergen,Hardenberg,Mariënberg","GBG, HDB, MRB",defecte trein,broken down train,defecte trein,broken down train,rolling stock,2021-01-01 12:25:19,2021-01-01 12:45:00,20.0
4,35630,Roosendaal-Vlissingen,Roosendaal - Vlissingen,21,"Bergen op Zoom,Rilland-Bath","BGN, RB",aanrijding,collision,aanrijding,collision,accidents,2021-01-01 15:17:02,2021-01-01 15:58:59,42.0


In [ ]:
# Split the text "HGL, HGLO, ODZ" into a list ["HGL", " HGLO", " ODZ"] casue we want to give every station in the list its own row
storingen['station_code'] = storingen['rdt_station_codes'].str.split(',')

# Give every station in the list its own row so that we can join the disruptions to the stations
storingen = storingen.explode('station_code')

# Remove the spaces around the codes (" HGLO" becomes "HGLO") otherwise the join with the stations will not work
storingen['station_code'] = storingen['station_code'].str.strip()

print(len(storingen))

#checking compared to first first few rows of the dataframe. if now every row had a station. 
storingen.head()

439242


,rdt_id,ns_lines,rdt_lines,rdt_lines_id,rdt_station_names,rdt_station_codes,cause_nl,cause_en,statistical_cause_nl,statistical_cause_en,cause_group,start_time,end_time,duration_minutes,station_code
0,35626,Hengelo-Bielefeld,"Berlin Ostbahnhof - Hengelo, Bielefeld Hbf - H...","91,124,174","Hengelo,Hengelo Oost,Oldenzaal","HGL, HGLO, ODZ",defecte trein,broken down train,defecte trein,broken down train,rolling stock,2021-01-01 08:12:55,2021-01-01 11:30:00,197.0,HGL
0,35626,Hengelo-Bielefeld,"Berlin Ostbahnhof - Hengelo, Bielefeld Hbf - H...","91,124,174","Hengelo,Hengelo Oost,Oldenzaal","HGL, HGLO, ODZ",defecte trein,broken down train,defecte trein,broken down train,rolling stock,2021-01-01 08:12:55,2021-01-01 11:30:00,197.0,HGLO
0,35626,Hengelo-Bielefeld,"Berlin Ostbahnhof - Hengelo, Bielefeld Hbf - H...","91,124,174","Hengelo,Hengelo Oost,Oldenzaal","HGL, HGLO, ODZ",defecte trein,broken down train,defecte trein,broken down train,rolling stock,2021-01-01 08:12:55,2021-01-01 11:30:00,197.0,ODZ
0,35626,Hengelo-Bielefeld,"Berlin Ostbahnhof - Hengelo, Bielefeld Hbf - H...","91,124,174","Hengelo,Hengelo Oost,Oldenzaal","HGL, HGLO, ODZ",defecte trein,broken down train,defecte trein,broken down train,rolling stock,2021-01-01 08:12:55,2021-01-01 11:30:00,197.0,HGL
0,35626,Hengelo-Bielefeld,"Berlin Ostbahnhof - Hengelo, Bielefeld Hbf - H...","91,124,174","Hengelo,Hengelo Oost,Oldenzaal","HGL, HGLO, ODZ",defecte trein,broken down train,defecte trein,broken down train,rolling stock,2021-01-01 08:12:55,2021-01-01 11:30:00,197.0,HGLO


In [6]:
# Keep only station codes that are in the Dutch station list
storingen = storingen[storingen['station_code'].isin(stations['code'])]

print(len(storingen))

426093


In [ ]:
#calculation For each station we calculate:the average number of disruptions per year,the average number of disruption minutes per year
#We divide by 3 because the data covers three years (2021, 2022 and 2023).

# Count the number of disruptions per station
aantal = storingen.groupby('station_code')['rdt_id'].count()

# Add up the disruption minutes per station
minuten = storingen.groupby('station_code')['duration_minutes'].sum()

# Put both in one table and divide by 3 to get the average per year
per_station = pd.DataFrame({
    'storingen_per_jaar': aantal / 3,
    'minuten_per_jaar': minuten / 3,
})
per_station = per_station.reset_index()

per_station.head()

,station_code,storingen_per_jaar,minuten_per_jaar
0,AC,836.000000,248968.333333
1,AH,1588.666667,436684.333333
2,AHP,854.666667,252817.666667
3,AHPR,260.666667,18460.333333
4,AHZ,517.333333,147056.666667


In [8]:
# Start from the full list of stations
alle_stations = stations[['code']].rename(columns={'code': 'station_code'})

# Add the disruption numbers
storingen_per_station = alle_stations.merge(per_station, on='station_code', how='left')

# Stations without disruptions get 0 instead of an empty value
storingen_per_station = storingen_per_station.fillna(0)

print(len(storingen_per_station))
storingen_per_station.sort_values('storingen_per_jaar', ascending=False).head(10)

397


,station_code,storingen_per_jaar,minuten_per_jaar
330,UT,2764.666667,601590.000000
288,RTD,2731.666667,526868.000000
303,SHL,2262.000000,629396.333333
22,ASD,1987.333333,616053.333333
37,AH,1588.666667,436684.333333
119,DVD,1544.333333,436980.333333
111,DDR,1518.333333,282977.000000
226,LEDN,1458.333333,284749.333333
28,ASS,1390.333333,373102.333333
94,GVC,1343.000000,280893.333333


In [9]:
storingen_per_station.to_pickle(PROCESSED / 'storingen_per_station.pkl')